# Semana 01 — Lab 01: Baseline + split + métricas (solo con NumPy)

## Contexto
En los slides vimos:
- **Generalización** y por qué necesitamos evaluar fuera de entrenamiento
- **Fuga de información** (data leakage)
- **Modelos de referencia** (baselines)
- **Métricas**: matriz de confusión, exactitud (accuracy), precisión (precision), recall
- **Curva PR** y **AP** (Average Precision) como resumen cuando hay desbalance

## Objetivo del laboratorio
Implementar el workflow **sin magia** (con NumPy), para entender exactamente:
- cómo se hace un split reproducible train/val/test
- cómo se construye un baseline
- cómo cambian precisión/recall al variar un umbral
- cómo aparece la fuga por preprocesamiento

## Entregable
En una celda final incluye:
- una **tabla** de umbrales con (accuracy, precision, recall) en validación
- una **gráfica PR** (recall vs precision)
- una **comparación** “con fuga vs sin fuga” en el demo de estandarización
- 3–5 líneas de comentario (qué aprendiste y qué decisión de umbral tomarías)

> Nota: aquí no usamos scikit-learn. Solo NumPy y Matplotlib.

## 0) Setup

- Usaremos un generador `rng` para que todo sea **reproducible**.
- Implementaremos utilidades mínimas para:
  - matriz de confusión (TP, FP, FN, TN)
  - métricas: accuracy, precision, recall

> Convención: clase positiva = `y=1`.

In [ ]:
# 1) Imports y semilla (reproducible)

import numpy as np
import matplotlib.pyplot as plt

# Reproducibilidad
rng = np.random.default_rng(0)

# Estilo simple para plots
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
})


## 1) Utilidades mínimas: matriz de confusión y métricas

Implementaremos todo con NumPy.

- Matriz de confusión: \(TP, FP, FN, TN\)
- Métricas: exactitud (accuracy), precisión (precision), recall

> Convención: clase positiva = `y=1`.


In [ ]:
def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray):
    """Regresa (TP, FP, FN, TN) para clasificación binaria con y∈{0,1}."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn


def accuracy_from_counts(tp, fp, fn, tn):
    denom = tp + fp + fn + tn
    return (tp + tn) / denom if denom else np.nan


def precision_from_counts(tp, fp, fn, tn):
    denom = tp + fp
    return tp / denom if denom else 0.0


def recall_from_counts(tp, fp, fn, tn):
    denom = tp + fn
    return tp / denom if denom else 0.0


def metrics(y_true: np.ndarray, y_pred: np.ndarray):
    """Regresa dict con TP,FP,FN,TN y métricas principales."""
    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "accuracy": accuracy_from_counts(tp, fp, fn, tn),
        "precision": precision_from_counts(tp, fp, fn, tn),
        "recall": recall_from_counts(tp, fp, fn, tn),
    }


# Mini-test rápido
y_true = np.array([1, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 1, 0])
metrics(y_true, y_pred)


## 2) Dataset sintético binario (desbalanceado)

Generaremos dos nubes gaussianas en \(\mathbb{R}^2\) con desbalance (pocos positivos).

- `y=1` (positivos): minoría
- `y=0` (negativos): mayoría

Esto nos permite ver por qué **accuracy** puede engañar y por qué la curva **Precisión–Recall** es útil.


In [ ]:
# Tamaños (desbalance)
n_pos = 80
n_neg = 520
n = n_pos + n_neg

# Distribuciones (elige medias/covarianzas simples)
mu_pos = np.array([2.0, 1.5])
mu_neg = np.array([0.0, 0.0])

Sigma_pos = np.array([[1.0, 0.2], [0.2, 1.0]])
Sigma_neg = np.array([[1.2, -0.1], [-0.1, 1.0]])

X_pos = rng.multivariate_normal(mu_pos, Sigma_pos, size=n_pos)
X_neg = rng.multivariate_normal(mu_neg, Sigma_neg, size=n_neg)

X = np.vstack([X_pos, X_neg])
y = np.hstack([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)])

# Barajamos para mezclar clases
perm = rng.permutation(n)
X = X[perm]
y = y[perm]

# Visual rápido
plt.figure()
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=14, alpha=0.6, label="y=0 (neg)")
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=18, alpha=0.9, label="y=1 (pos)")
plt.title("Dataset sintético 2D (desbalanceado)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

# Prevalencia
pi = np.mean(y == 1)
pi


## 3) Split reproducible train/val/test (60/20/20)

Regla práctica: todo lo que “aprendas” (parámetros de modelos, umbrales, transformaciones) se decide usando solo **train** (y **val** para seleccionar), y el **test** se guarda para auditoría final.


In [ ]:
def split_train_val_test(X, y, frac_train=0.6, frac_val=0.2, seed=0):
    """Split reproducible usando permutación de índices."""
    rng_local = np.random.default_rng(seed)
    n = len(y)
    idx = rng_local.permutation(n)

    n_train = int(frac_train * n)
    n_val = int(frac_val * n)

    idx_train = idx[:n_train]
    idx_val = idx[n_train:n_train + n_val]
    idx_test = idx[n_train + n_val:]

    return (X[idx_train], y[idx_train]), (X[idx_val], y[idx_val]), (X[idx_test], y[idx_test])


(Xtr, ytr), (Xva, yva), (Xte, yte) = split_train_val_test(X, y, 0.6, 0.2, seed=42)

print("n_train, n_val, n_test:", len(ytr), len(yva), len(yte))
print("prevalencia (train/val/test):", np.mean(ytr), np.mean(yva), np.mean(yte))


## 4) Modelo de referencia (baseline): clase mayoritaria

Un baseline obligatorio para clasificación binaria es predecir siempre la clase mayoritaria en **train**.

Esto suele dar una exactitud alta cuando hay desbalance, pero puede tener recall = 0.


In [ ]:
# Baseline: clase mayoritaria en train
majority_class = int(np.round(np.mean(ytr)))  # si prevalencia>0.5 -> 1, si no -> 0
# (Como es desbalanceado hacia 0, típicamente majority_class = 0)

pred_val = np.full_like(yva, majority_class)
pred_test = np.full_like(yte, majority_class)

m_val = metrics(yva, pred_val)
m_test = metrics(yte, pred_test)

print("Baseline (mayoritaria) — val:", m_val)
print("Baseline (mayoritaria) — test:", m_test)


## 5) Score + umbral: construimos un clasificador “mínimo”

En vez de entrenar un modelo, definimos un score simple \(s(x)\). Por ejemplo:

\[
 s(x)=w^\top x \quad \text{con un } w \text{ fijo.}
\]

Luego predecimos:
\[
\hat y = \mathbb{1}[s(x)\ge t]
\]

La idea del lab es ver cómo cambian **precisión/recall** al variar el umbral \(t\).


In [ ]:
# Score lineal fijo (no entrenamos): w elegido "a mano"
w = np.array([1.0, 0.7])

def score(X):
    return X @ w

s_tr = score(Xtr)
s_va = score(Xva)
s_te = score(Xte)

# Visual del score en train
plt.figure()
plt.hist(s_tr[ytr == 0], bins=30, alpha=0.6, label="y=0")
plt.hist(s_tr[ytr == 1], bins=30, alpha=0.7, label="y=1")
plt.title("Distribución del score s(x) en train")
plt.xlabel("s(x)")
plt.ylabel("conteo")
plt.legend()
plt.show()


## 6) Barrido de umbral (en validación) + tabla de métricas

Usaremos **validación** para elegir umbral (no test).

- Barreremos varios valores de \(t\)
- Calcularemos exactitud, precisión y recall
- Guardaremos los puntos para graficar la curva PR


In [ ]:
def sweep_thresholds(y_true, scores, thresholds):
    rows = []
    for t in thresholds:
        y_pred = (scores >= t).astype(int)
        m = metrics(y_true, y_pred)
        rows.append((
            float(t),
            m["accuracy"],
            m["precision"],
            m["recall"],
            m["TP"], m["FP"], m["FN"], m["TN"],
        ))
    return np.array(rows, dtype=float)

# Umbrales: tomamos cuantiles del score en validación
thresholds = np.quantile(s_va, np.linspace(0.0, 1.0, 25))
rows = sweep_thresholds(yva, s_va, thresholds)

# Ordenamos por umbral (de alto a bajo) para lectura
rows_sorted = rows[np.argsort(rows[:, 0])[::-1]]

# Tabla (t, acc, prec, rec)
print("t\t\tacc\tprec\trec")
for t, acc, prec, rec, *_ in rows_sorted:
    print(f"{t: .3f}\t{acc: .3f}\t{prec: .3f}\t{rec: .3f}")

# Elegimos un umbral con un criterio simple (ejemplo): maximizar F1 en validación
prec = rows[:, 2]
rec = rows[:, 3]
f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)
idx_best = int(np.argmax(f1))

t_star = rows[idx_best, 0]
print("\nUmbral elegido (max F1 en val):", t_star)
print("Métricas en val con t*:", metrics(yva, (s_va >= t_star).astype(int)))


## 7) Curva Precisión–Recall (PR)

Con los puntos del barrido, graficamos:
- eje x: **recall**
- eje y: **precisión**

Recuerda: la línea base no-informativa tiene precisión ≈ prevalencia \(\pi=P(y=1)\).


In [ ]:
# Puntos PR desde el barrido
prec_pts = rows[:, 2]
rec_pts = rows[:, 3]

# Para visualizar bonito: ordenamos por recall
order = np.argsort(rec_pts)
rec_plot = rec_pts[order]
prec_plot = prec_pts[order]

plt.figure()
plt.plot(rec_plot, prec_plot, marker="o", linewidth=2)
plt.axhline(pi, linestyle="--", color="black", linewidth=1.5, label=f"línea base ≈ prevalencia π={pi:.2f}")
plt.xlabel("recall")
plt.ylabel("precisión")
plt.title("Curva Precisión–Recall (desde barrido de umbral)")
plt.legend()
plt.show()


## 8) Auditoría final en test (sin volver a elegir nada)

Usamos el umbral \(t^\*\) elegido en validación y reportamos métricas en test.

> Si empiezas a “ajustar” \(t\) mirando test, introduces sesgo por selección.


In [ ]:
yhat_test = (s_te >= t_star).astype(int)
print("Métricas en test con t* (elegido en val):")
print(metrics(yte, yhat_test))


## 9) Mini-demostración de fuga por preprocesamiento (estandarización)

Vamos a simular un escenario “tipo temporal”:
- train/val vienen del "presente"
- test viene del "futuro" con un cambio de distribución (drift)

Luego comparamos dos pipelines:
- **Con fuga**: estandarizar ajustando media/desv con train+val+test
- **Sin fuga**: estandarizar ajustando solo con train

**Importante**: el modelo y el umbral se eligen con validación, pero el *fit* del escalador no debe ver test.


In [ ]:
def standardize_fit(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    return mu, sd


def standardize_transform(X, mu, sd):
    return (X - mu) / sd

# Datos "temporales": el test tiene drift en x1
rng2 = np.random.default_rng(123)

n_train, n_val, n_test = 240, 80, 80
Xtr2 = rng2.normal(0, 1, size=(n_train, 2))
Xva2 = rng2.normal(0, 1, size=(n_val, 2))
Xte2 = rng2.normal(0, 1, size=(n_test, 2))

# Drift SOLO en test
Xte2[:, 0] += 3.0

# Etiquetas: función del feature 0 con ruido (para que no sea separable perfecto)
noise_tr = rng2.normal(0, 0.7, size=n_train)
noise_va = rng2.normal(0, 0.7, size=n_val)
noise_te = rng2.normal(0, 0.7, size=n_test)

ytr2 = ((Xtr2[:, 0] + 0.3 * Xtr2[:, 1] + noise_tr) > 0.8).astype(int)
yva2 = ((Xva2[:, 0] + 0.3 * Xva2[:, 1] + noise_va) > 0.8).astype(int)
yte2 = ((Xte2[:, 0] + 0.3 * Xte2[:, 1] + noise_te) > 0.8).astype(int)

# Score fijo (igual idea que antes): usamos x1 estandarizada como score

def run_pipeline(mu_fit_source: str):
    if mu_fit_source == "LEAK":
        X_all = np.vstack([Xtr2, Xva2, Xte2])
        mu, sd = standardize_fit(X_all)  # <-- fuga: ve test
    elif mu_fit_source == "OK":
        mu, sd = standardize_fit(Xtr2)   # <-- correcto: solo train
    else:
        raise ValueError

    Ztr = standardize_transform(Xtr2, mu, sd)
    Zva = standardize_transform(Xva2, mu, sd)
    Zte = standardize_transform(Xte2, mu, sd)

    s_va2 = Zva[:, 0]
    s_te2 = Zte[:, 0]

    # Elegimos umbral con validación (max F1)
    thr = np.quantile(s_va2, np.linspace(0.0, 1.0, 25))
    rows2 = sweep_thresholds(yva2, s_va2, thr)
    prec2, rec2 = rows2[:, 2], rows2[:, 3]
    f12 = np.where((prec2 + rec2) > 0, 2 * prec2 * rec2 / (prec2 + rec2), 0.0)
    t2 = rows2[int(np.argmax(f12)), 0]

    out = {
        "t_star": float(t2),
        "val": metrics(yva2, (s_va2 >= t2).astype(int)),
        "test": metrics(yte2, (s_te2 >= t2).astype(int)),
        "mu": mu,
        "sd": sd,
    }
    return out

res_leak = run_pipeline("LEAK")
res_ok = run_pipeline("OK")

print("=== Sin fuga (fit escalador solo train) ===")
print("t*:", res_ok["t_star"], "val:", res_ok["val"], "test:", res_ok["test"])
print("\n=== Con fuga (fit escalador con train+val+test) ===")
print("t*:", res_leak["t_star"], "val:", res_leak["val"], "test:", res_leak["test"])

# Visual: cómo cambia la nube estandarizada
mu_ok, sd_ok = res_ok["mu"], res_ok["sd"]
mu_leak, sd_leak = res_leak["mu"], res_leak["sd"]

Zte_ok = standardize_transform(Xte2, mu_ok, sd_ok)
Zte_leak = standardize_transform(Xte2, mu_leak, sd_leak)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(Zte_ok[:, 0], Zte_ok[:, 1], s=14)
plt.title("Test estandarizado\n(sin fuga)")
plt.xlim(-4, 4)
plt.ylim(-4, 4)

plt.subplot(1, 2, 2)
plt.scatter(Zte_leak[:, 0], Zte_leak[:, 1], s=14)
plt.title("Test estandarizado\n(con fuga)")
plt.xlim(-4, 4)
plt.ylim(-4, 4)

plt.tight_layout()
plt.show()


## 10) Celda final (entregable)

Copia/pega aquí tus resultados finales (con números concretos de tu ejecución):

- **Tabla de umbrales en validación** (la que imprimiste en el barrido)
- **Curva PR** (recall vs precisión)
- **Comparación con fuga vs sin fuga** (métricas en test)

### Comentario (3–5 líneas)
- ¿Qué métrica usarías si el positivo es raro?
- ¿Qué umbral elegirías y por qué (precision vs recall)?
- ¿Qué aprendiste de la fuga por estandarización?
